# 02. Spatial Preprocessing
**목적**: 기상 데이터와 전력설비 데이터를 공간 객체로 변환하고, 좌표계를 통일한 뒤 공간 매칭을 수행한다.

단계:
1. column_map.json 로드
2. 전력설비 GeoDataFrame 생성 (point 또는 line)
3. 기상 관측소 GeoDataFrame 생성
4. 좌표계 변환 (WGS84 → UTM-K)
5. 기상-설비 공간 매칭 (최근접 + IDW)
6. 전처리 결과 저장

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

from config import (DATA_RAW, DATA_PROCESSED, CRS_GEO, CRS_PROJ,
                    SPATIAL_MATCH_METHOD, IDW_POWER)

In [ ]:
with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)

FLAGS = cmap['flags']
WC = cmap['weather']     # weather column map
FC = cmap['facility']    # facility column map

print('모델 유형:', 'Case A (Supervised)' if FLAGS['has_label'] else 'Case B (Unsupervised)')
print('설비 형태:', FLAGS['facility_geometry'])

## 1. 데이터 로드

In [ ]:
# ── 파일명 수정 ──
df_w = pd.read_csv(DATA_RAW / 'weather.csv', encoding='utf-8-sig')
df_f = pd.read_csv(DATA_RAW / 'facility.csv', encoding='utf-8-sig')

print(f'기상 데이터: {df_w.shape}  |  전력설비: {df_f.shape}')

## 2. 전력설비 GeoDataFrame 생성

In [ ]:
LAT_COL = FC['lat']   # 실제 위도 컬럼명
LON_COL = FC['lon']   # 실제 경도 컬럼명
FID_COL = FC['facility_id']

if FLAGS['facility_geometry'] == 'point':
    geometry = gpd.points_from_xy(df_f[LON_COL], df_f[LAT_COL])
    gdf_fac = gpd.GeoDataFrame(df_f, geometry=geometry, crs=CRS_GEO)
else:
    # line 형태: from/to 좌표 또는 WKT 컬럼이 있는 경우
    # 실제 데이터 구조에 따라 수정 필요
    raise NotImplementedError("Line geometry — 실제 데이터 구조 확인 후 구현")

print(f'설비 GeoDataFrame: {len(gdf_fac):,}개  |  CRS: {gdf_fac.crs}')
gdf_fac.head()

## 3. 기상 관측소 GeoDataFrame 생성

In [ ]:
STN_LAT = WC['lat']   # 관측소 위도 컬럼명
STN_LON = WC['lon']   # 관측소 경도 컬럼명
STN_ID  = WC['station_id']

# 관측소 좌표 (중복 제거)
df_stn = df_w[[STN_ID, STN_LAT, STN_LON]].drop_duplicates(subset=STN_ID).reset_index(drop=True)

gdf_stn = gpd.GeoDataFrame(
    df_stn,
    geometry=gpd.points_from_xy(df_stn[STN_LON], df_stn[STN_LAT]),
    crs=CRS_GEO
)

print(f'관측소 수: {len(gdf_stn)}')
gdf_stn.head()

## 4. 좌표계 변환 (WGS84 → UTM-K)
buffer 생성 및 거리 계산은 미터 기반 좌표계에서 수행한다.

In [ ]:
gdf_fac_proj = gdf_fac.to_crs(CRS_PROJ)
gdf_stn_proj = gdf_stn.to_crs(CRS_PROJ)

print(f'설비 CRS: {gdf_fac_proj.crs}')
print(f'관측소 CRS: {gdf_stn_proj.crs}')

## 5. 기상-설비 공간 매칭

In [ ]:
def nearest_station_match(gdf_fac_proj, gdf_stn_proj, stn_id_col):
    """각 설비에 가장 가까운 관측소 ID와 거리를 매핑한다."""
    fac_coords = np.column_stack([
        gdf_fac_proj.geometry.x,
        gdf_fac_proj.geometry.y
    ])
    stn_coords = np.column_stack([
        gdf_stn_proj.geometry.x,
        gdf_stn_proj.geometry.y
    ])

    tree = cKDTree(stn_coords)
    dist, idx = tree.query(fac_coords, k=1)

    result = gdf_fac_proj[[FC['facility_id']]].copy()
    result['nearest_stn_id'] = gdf_stn_proj[stn_id_col].values[idx]
    result['nearest_stn_dist_m'] = dist
    return result


def idw_station_match(gdf_fac_proj, gdf_stn_proj, stn_id_col, k=5, power=IDW_POWER):
    """가까운 k개 관측소 ID와 IDW 가중치를 매핑한다 (민감도 분석용)."""
    fac_coords = np.column_stack([gdf_fac_proj.geometry.x, gdf_fac_proj.geometry.y])
    stn_coords = np.column_stack([gdf_stn_proj.geometry.x, gdf_stn_proj.geometry.y])

    tree = cKDTree(stn_coords)
    dists, idxs = tree.query(fac_coords, k=k)

    # 거리 0 처리 (설비와 관측소가 동일 위치인 경우)
    dists = np.where(dists == 0, 1e-6, dists)
    weights = (1.0 / dists ** power)
    weights = weights / weights.sum(axis=1, keepdims=True)

    # facility_id → [(stn_id, weight), ...] 딕셔너리
    fac_ids = gdf_fac_proj[FC['facility_id']].values
    idw_map = {}
    for i, fid in enumerate(fac_ids):
        idw_map[fid] = [
            (gdf_stn_proj[stn_id_col].values[idxs[i, j]], weights[i, j])
            for j in range(k)
        ]
    return idw_map

In [ ]:
# 기본 매칭 수행
df_match = nearest_station_match(gdf_fac_proj, gdf_stn_proj, STN_ID)

print(f'매칭 완료: {len(df_match):,}개 설비')
print(f'최근접 관측소 거리 — 중앙값: {df_match.nearest_stn_dist_m.median():.0f}m  '
      f'최대: {df_match.nearest_stn_dist_m.max():.0f}m')
df_match.head()

In [ ]:
# IDW 매핑 (민감도 분석용, 저장해두기)
idw_map = idw_station_match(gdf_fac_proj, gdf_stn_proj, STN_ID, k=5)

import pickle
with open(DATA_PROCESSED / 'idw_map.pkl', 'wb') as f:
    pickle.dump(idw_map, f)

print('IDW 매핑 저장 완료')

## 6. 결과 저장

In [ ]:
# 설비 GeoDataFrame + 매칭 결과 병합
gdf_fac_proj = gdf_fac_proj.merge(df_match, on=FC['facility_id'], how='left')

# WGS84 버전도 보존
gdf_fac_geo = gdf_fac_proj.to_crs(CRS_GEO)

gdf_fac_proj.to_file(DATA_PROCESSED / 'facility_proj.gpkg', driver='GPKG')
gdf_fac_geo.to_file(DATA_PROCESSED / 'facility_geo.gpkg', driver='GPKG')
gdf_stn_proj.to_file(DATA_PROCESSED / 'stations_proj.gpkg', driver='GPKG')

print('저장 완료')
print('  facility_proj.gpkg — UTM-K (buffer 계산용)')
print('  facility_geo.gpkg  — WGS84 (지도 시각화용)')
print('  stations_proj.gpkg — 관측소 (UTM-K)')
print('\n다음 단계: 03_buffer_feature_engineering.ipynb')